In [0]:
# Check the Spark environment
print("AI Demand Forecasting Project")

print("Spark version:", spark.version)

# Create a small sample sales dataset
sales_data = [
    ("2026-10-01", "P001", "Rice", 25, 60.0),
    ("2026-10-02", "P001", "Rice", 30, 60.0),
    ("2026-10-03", "P001", "Rice", 42, 58.0),
    ("2026-10-01", "P002", "Cooking Oil", 18, 140.0),
    ("2026-10-02", "P002", "Cooking Oil", 22, 140.0),
    ("2026-10-03", "P002", "Cooking Oil", 28, 135.0)
]

columns = [
    "sale_date",
    "product_id",
    "product_name",
    "units_sold",
    "unit_price"
]

sales_df = spark.createDataFrame(sales_data, columns)

display(sales_df)

AI Demand Forecasting Project
Spark version: 4.2.0


sale_date,product_id,product_name,units_sold,unit_price
2026-10-01,P001,Rice,25,60.0
2026-10-02,P001,Rice,30,60.0
2026-10-03,P001,Rice,42,58.0
2026-10-01,P002,Cooking Oil,18,140.0
2026-10-02,P002,Cooking Oil,22,140.0
2026-10-03,P002,Cooking Oil,28,135.0


In [0]:
#Calculate the revenue 

from pyspark.sql.functions import col

sales_df = sales_df.withColumn(
    "revenue",
    col("units_sold") * col("unit_price")
)

display(sales_df)

sale_date,product_id,product_name,units_sold,unit_price,revenue
2026-10-01,P001,Rice,25,60.0,1500.0
2026-10-02,P001,Rice,30,60.0,1800.0
2026-10-03,P001,Rice,42,58.0,2436.0
2026-10-01,P002,Cooking Oil,18,140.0,2520.0
2026-10-02,P002,Cooking Oil,22,140.0,3080.0
2026-10-03,P002,Cooking Oil,28,135.0,3780.0




# Step 3: Data Preparation for AI Demand Forecasting

## Objective
Prepare the raw sales data for machine learning by converting dates
into the correct data type and creating useful time-based features.

## Why is this important?
Machine learning models work better when dates and numerical
features are properly prepared. These features help us understand
how product demand changes over time.

## Tasks
1. Convert the sales date into a date format.
2. Extract day, month, and day of the week.
3. Sort the data chronologically.
4. Prepare the dataset for future demand forecasting.

In [0]:

from pyspark.sql.functions import sum, avg, max, min, count

# Analyze sales for each product
product_summary = sales_df.groupBy("product_name").agg(
    sum("units_sold").alias("total_units_sold"),
    sum("revenue").alias("total_revenue"),
    avg("units_sold").alias("average_daily_sales"),
    max("units_sold").alias("maximum_daily_sales"),
    min("units_sold").alias("minimum_daily_sales"),
    count("*").alias("number_of_records")
)

# Display the summary
display(product_summary)


product_name,total_units_sold,total_revenue,average_daily_sales,maximum_daily_sales,minimum_daily_sales,number_of_records
Rice,97,5736.0,32.333333333333336,42,25,3
Cooking Oil,68,9380.0,22.666666666666668,28,18,3


In [0]:

# ============================================================
# STEP 3: DATA PREPARATION
# Project: AI Demand Forecasting & Inventory Optimization
# Tool: Databricks + PySpark
# Purpose: Prepare sales data for machine learning
# ============================================================

# Import functions required for date processing
from pyspark.sql.functions import (
    col,
    to_date,
    dayofmonth,
    month,
    dayofweek
)

# Convert sale_date from string to a proper date type
# This allows us to perform date-based analysis correctly.
sales_prepared_df = sales_df.withColumn(
    "sale_date",
    to_date(col("sale_date"), "yyyy-MM-dd")
)

# Extract the day of the month from the sales date
sales_prepared_df = sales_prepared_df.withColumn(
    "day_of_month",
    dayofmonth(col("sale_date"))
)

# Extract the month to identify monthly sales patterns
sales_prepared_df = sales_prepared_df.withColumn(
    "month",
    month(col("sale_date"))
)

# Extract the day of the week
# Spark uses 1 for Sunday and 7 for Saturday.
sales_prepared_df = sales_prepared_df.withColumn(
    "day_of_week",
    dayofweek(col("sale_date"))
)

# Sort the records by product and date
# Chronological ordering makes the dataset easier to inspect.
sales_prepared_df = sales_prepared_df.orderBy(
    "product_id",
    "sale_date"
)

# Display the prepared dataset
display(sales_prepared_df)


sale_date,product_id,product_name,units_sold,unit_price,revenue,day_of_month,month,day_of_week
2026-10-01,P001,Rice,25,60.0,1500.0,1,10,5
2026-10-02,P001,Rice,30,60.0,1800.0,2,10,6
2026-10-03,P001,Rice,42,58.0,2436.0,3,10,7
2026-10-01,P002,Cooking Oil,18,140.0,2520.0,1,10,5
2026-10-02,P002,Cooking Oil,22,140.0,3080.0,2,10,6
2026-10-03,P002,Cooking Oil,28,135.0,3780.0,3,10,7


In [0]:

# ============================================================
# STEP 4: DATA QUALITY CHECKS
# ============================================================

from pyspark.sql.functions import count, when

# Display column names and data types
# This verifies that sale_date is now a date rather than a string.
sales_prepared_df.printSchema()

# Count missing values in each column
# Null values can cause problems during model training.
missing_values = sales_prepared_df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in sales_prepared_df.columns
])

display(missing_values)

# Check for duplicate records for the same product and date.
# Each product should have one daily sales record in this dataset.
duplicate_records = (
    sales_prepared_df
    .groupBy("product_id", "sale_date")
    .count()
    .filter(col("count") > 1)
)

display(duplicate_records)


root
 |-- sale_date: date (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- units_sold: long (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- day_of_month: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)



sale_date,product_id,product_name,units_sold,unit_price,revenue,day_of_month,month,day_of_week
0,0,0,0,0,0,0,0,0


product_id,sale_date,count



# Step 5: Generate Historical Sales Data

## Business Objective
Create a realistic historical sales dataset for a retail business
to support AI-powered demand forecasting and inventory optimization.

## Dataset Design
- Historical period: October 2025 to September 2026
- Products: Rice, Cooking Oil, Wheat Flour, Sugar, and Milk
- Frequency: Daily sales
- Expected records: 1,825 rows

## Demand Patterns
The dataset includes simulated daily demand fluctuations,
weekend effects, gradual growth, and seasonal variation.

## Business Value
Historical sales patterns will help us train a machine learning
model to forecast future demand and make inventory decisions.

Note: This is synthetic data generated for project development,
not actual sales from a real business.


In [0]:

# ============================================================
# STEP 5: GENERATE HISTORICAL SALES DATA
# Project: AI Demand Forecasting & Inventory Optimization
# Platform: Databricks | Language: PySpark
# ============================================================

from pyspark.sql.functions import (
    col, lit, expr, sequence, explode,
    dayofweek, dayofyear, datediff,
    sin, rand, round, greatest
)

# ------------------------------------------------------------
# 1. Define the product catalog
# Each product has a unique ID, price, and average daily demand.
# ------------------------------------------------------------

products = [
    ("P001", "Rice", 55.0, 30.0),
    ("P002", "Cooking Oil", 150.0, 20.0),
    ("P003", "Wheat Flour", 45.0, 25.0),
    ("P004", "Sugar", 48.0, 18.0),
    ("P005", "Milk", 30.0, 40.0)
]

products_df = spark.createDataFrame(
    products,
    ["product_id", "product_name", "unit_price", "base_demand"]
)

# ------------------------------------------------------------
# 2. Create a calendar containing 365 consecutive days.
# ------------------------------------------------------------

calendar_df = spark.range(1).select(
    explode(
        sequence(
            lit("2025-10-01").cast("date"),
            lit("2026-09-30").cast("date"),
            expr("INTERVAL 1 DAY")
        )
    ).alias("sale_date")
)

# ------------------------------------------------------------
# 3. Combine every date with every product.
# This creates one record per product per day.
# 365 days x 5 products = 1,825 records.
# ------------------------------------------------------------

sales_history_df = calendar_df.crossJoin(products_df)

# ------------------------------------------------------------
# 4. Create realistic demand patterns.
# - Weekend effect: slightly higher demand on weekends.
# - Seasonal effect: demand changes across the year.
# - Growth effect: gradual demand growth over time.
# - Random variation: daily demand is not always identical.
# ------------------------------------------------------------

sales_history_df = sales_history_df.withColumn(
    "weekend_effect",
    expr(
        "CASE WHEN dayofweek(sale_date) IN (1, 7) "
        "THEN 0.12 ELSE 0.0 END"
    )
)

sales_history_df = sales_history_df.withColumn(
    "seasonal_effect",
    0.15 * sin(
        dayofyear(col("sale_date")) * (2.0 * 3.14159265 / 365.0)
    )
)

sales_history_df = sales_history_df.withColumn(
    "growth_effect",
    datediff(col("sale_date"), lit("2025-10-01").cast("date"))
    / 365.0 * 0.10
)

sales_history_df = sales_history_df.withColumn(
    "random_effect",
    (rand(seed=42) - 0.5) * 0.30
)

# Calculate daily units sold using the combined demand effects.
sales_history_df = sales_history_df.withColumn(
    "units_sold",
    greatest(
        lit(1),
        round(
            col("base_demand") * (
                1
                + col("weekend_effect")
                + col("seasonal_effect")
                + col("growth_effect")
                + col("random_effect")
            ),
            0
        )
    ).cast("int")
)

# ------------------------------------------------------------
# 5. Calculate revenue and select final business columns.
# Revenue = units sold x unit price.
# ------------------------------------------------------------

sales_history_df = sales_history_df.withColumn(
    "revenue",
    round(col("units_sold") * col("unit_price"), 2)
)

sales_history_df = sales_history_df.select(
    "sale_date",
    "product_id",
    "product_name",
    "units_sold",
    "unit_price",
    "revenue"
).orderBy("sale_date", "product_id")

# ------------------------------------------------------------
# 6. Validate and display the generated dataset.
# ------------------------------------------------------------

print("Total records:", sales_history_df.count())
print("Total products:", products_df.count())

display(sales_history_df)


Total records: 1825
Total products: 5


sale_date,product_id,product_name,units_sold,unit_price,revenue
2025-10-01,P001,Rice,22,55.0,1210.0
2025-10-01,P002,Cooking Oil,17,150.0,2550.0
2025-10-01,P003,Wheat Flour,23,45.0,1035.0
2025-10-01,P004,Sugar,17,48.0,816.0
2025-10-01,P005,Milk,39,30.0,1170.0
2025-10-02,P001,Rice,24,55.0,1320.0
2025-10-02,P002,Cooking Oil,18,150.0,2700.0
2025-10-02,P003,Wheat Flour,18,45.0,810.0
2025-10-02,P004,Sugar,15,48.0,720.0
2025-10-02,P005,Milk,31,30.0,930.0



# Step 6: Feature Engineering

## Objective
Create historical sales features that help the machine learning
model learn patterns in product demand.

## Features Created
- Day of week
- Month
- Weekend indicator
- Previous-day sales (lag 1)
- Previous-week sales (lag 7)
- Rolling 7-day average demand

## Business Value
These features help the model identify recent demand changes,
weekly patterns, and potential increases or decreases in sales.

Important: Historical features must be calculated separately
for each product to avoid mixing the demand of different products.


In [0]:

# ============================================================
# STEP 6: FEATURE ENGINEERING
# Project: AI Demand Forecasting & Inventory Optimization
# Purpose: Create features for machine learning
# ============================================================

from pyspark.sql import Window
from pyspark.sql.functions import (
    col, dayofweek, month, when,
    lag, avg, coalesce, lit
)

# Sort historical sales separately for each product.
# This ensures that previous sales belong to the same product.
product_window = (
    Window.partitionBy("product_id")
    .orderBy("sale_date")
)

# Define a seven-day historical window for each product.
# The current day is excluded from the rolling average.
rolling_window = product_window.rowsBetween(-7, -1)

# Extract calendar features from the sales date.
sales_features_df = (
    sales_history_df
    .withColumn("day_of_week", dayofweek(col("sale_date")))
    .withColumn("month", month(col("sale_date")))
    .withColumn(
        "is_weekend",
        when(dayofweek(col("sale_date")).isin(1, 7), 1)
        .otherwise(0)
    )
)

# Create lag features using previous sales records.
# Lag 1 represents the previous day's sales.
sales_features_df = sales_features_df.withColumn(
    "sales_lag_1",
    lag("units_sold", 1).over(product_window)
)

# Lag 7 represents sales from seven days earlier.
sales_features_df = sales_features_df.withColumn(
    "sales_lag_7",
    lag("units_sold", 7).over(product_window)
)

# Calculate average sales across the previous seven records.
# Excluding the current day helps prevent data leakage.
sales_features_df = sales_features_df.withColumn(
    "rolling_avg_7",
    avg("units_sold").over(rolling_window)
)

# Keep early records with missing history.
# We will handle them explicitly during model preparation.
display(sales_features_df)


sale_date,product_id,product_name,units_sold,unit_price,revenue,day_of_week,month,is_weekend,sales_lag_1,sales_lag_7,rolling_avg_7
2025-10-01,P001,Rice,22,55.0,1210.0,4,10,0,null,null,null
2025-10-02,P001,Rice,24,55.0,1320.0,5,10,0,22,null,22.0
2025-10-03,P001,Rice,22,55.0,1210.0,6,10,0,24,null,23.0
2025-10-04,P001,Rice,27,55.0,1485.0,7,10,1,22,null,22.666666666666668
2025-10-05,P001,Rice,25,55.0,1375.0,1,10,1,27,null,23.75
2025-10-06,P001,Rice,24,55.0,1320.0,2,10,0,25,null,24.0
2025-10-07,P001,Rice,22,55.0,1210.0,3,10,0,24,null,24.0
2025-10-08,P001,Rice,23,55.0,1265.0,4,10,0,22,22,23.714285714285715
2025-10-09,P001,Rice,29,55.0,1595.0,5,10,0,23,24,23.857142857142858
2025-10-10,P001,Rice,26,55.0,1430.0,6,10,0,29,22,24.571428571428573



# Step 7: Train the Demand Forecasting Model

## Objective
Train a machine learning model to predict product demand
using historical sales data and engineered features.

## Algorithm
Random Forest Regression using Apache Spark MLlib.

## Training Strategy
- Use older dates to train the model.
- Use more recent dates to test its predictions.
- Exclude records that do not have sufficient historical features.

## Evaluation
Measure forecasting performance using:
- MAE (Mean Absolute Error)
- RMSE (Root Mean Squared Error)

## Business Value
Demand predictions will help us estimate future product requirements
and make better inventory replenishment decisions.


In [0]:

# ============================================================
# STEP 7A: PREPARE TRAINING AND TEST DATA
# ============================================================

from pyspark.sql.functions import col, lit, when

# Select the features that the model will learn from.
# The target variable is units_sold.
feature_columns = [
    "day_of_week",
    "month",
    "is_weekend",
    "sales_lag_1",
    "sales_lag_7",
    "rolling_avg_7"
]

# Remove rows without sufficient historical sales information.
# The first seven days for each product do not have all features.
model_df = sales_features_df.dropna(
    subset=feature_columns + ["units_sold"]
)

# Split chronologically: older dates for training, newer dates for testing.
# This is more appropriate for forecasting than a random split.
training_df = model_df.filter(
    col("sale_date") < lit("2026-08-01").cast("date")
)

testing_df = model_df.filter(
    col("sale_date") >= lit("2026-08-01").cast("date")
)

# Display dataset sizes and date ranges.
print("Training records:", training_df.count())
print("Testing records:", testing_df.count())

print("Training date range:")
training_df.selectExpr(
    "min(sale_date) AS start_date",
    "max(sale_date) AS end_date"
).show()

print("Testing date range:")
testing_df.selectExpr(
    "min(sale_date) AS start_date",
    "max(sale_date) AS end_date"
).show()


Training records: 1485
Testing records: 305
Training date range:
+----------+----------+
|start_date|  end_date|
+----------+----------+
|2025-10-08|2026-07-31|
+----------+----------+

Testing date range:
+----------+----------+
|start_date|  end_date|
+----------+----------+
|2026-08-01|2026-09-30|
+----------+----------+



In [0]:

# ============================================================
# STEP 7B: TRAIN RANDOM FOREST REGRESSION MODEL
# ============================================================

from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor

# Combine input features into a single vector column.
# Spark MLlib models expect features in this format.
assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

# Configure the Random Forest regression algorithm.
# A fixed seed makes results reproducible.
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="units_sold",
    predictionCol="predicted_units_sold",
    numTrees=50,
    maxDepth=5,
    seed=42
)

# Build a pipeline to apply feature assembly and model training.
pipeline = Pipeline(stages=[assembler, rf])

# Train the model using historical training records only.
forecast_model = pipeline.fit(training_df)

print("Random Forest model training completed successfully!")


Random Forest model training completed successfully!


In [0]:

# ============================================================
# STEP 7C: EVALUATE FORECASTING PERFORMANCE
# ============================================================

from pyspark.ml.evaluation import RegressionEvaluator

# Generate predictions for the held-out testing period.
predictions_df = forecast_model.transform(testing_df)

# Display actual and predicted demand side by side.
display(
    predictions_df.select(
        "sale_date",
        "product_id",
        "product_name",
        "units_sold",
        "predicted_units_sold"
    ).orderBy("sale_date", "product_id")
)

# MAE: average absolute difference between actual and predicted sales.
mae_evaluator = RegressionEvaluator(
    labelCol="units_sold",
    predictionCol="predicted_units_sold",
    metricName="mae"
)

# RMSE: penalizes larger prediction errors more strongly.
rmse_evaluator = RegressionEvaluator(
    labelCol="units_sold",
    predictionCol="predicted_units_sold",
    metricName="rmse"
)

mae = mae_evaluator.evaluate(predictions_df)
rmse = rmse_evaluator.evaluate(predictions_df)

print(f"Mean Absolute Error (MAE): {mae:.2f} units")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} units")


sale_date,product_id,product_name,units_sold,predicted_units_sold
2026-08-01,P001,Rice,34,34.190560568367935
2026-08-01,P002,Cooking Oil,25,22.455946421565365
2026-08-01,P003,Wheat Flour,30,26.025604865802944
2026-08-01,P004,Sugar,18,21.21570794335191
2026-08-01,P005,Milk,43,42.813682651562004
2026-08-02,P001,Rice,34,34.2128814491473
2026-08-02,P002,Cooking Oil,20,23.633106908468786
2026-08-02,P003,Wheat Flour,32,29.581485925364017
2026-08-02,P004,Sugar,20,20.083682201298856
2026-08-02,P005,Milk,42,47.17783273206274


Mean Absolute Error (MAE): 2.38 units
Root Mean Squared Error (RMSE): 2.93 units



# Step 8: Forecast Visualization and Error Analysis

## Objective
Visualize model predictions and compare them with actual sales.

## Analysis
1. Compare actual and predicted demand over time.
2. Calculate forecasting errors for each product.
3. Identify products where the model performs better or worse.

## Business Value
Visual analysis helps business teams understand forecast accuracy
and identify products that may require better forecasting methods.


In [0]:

# ============================================================
# STEP 8A: PRODUCT-LEVEL FORECAST EVALUATION
# ============================================================

from pyspark.sql.functions import (
    abs, avg, sqrt, count, round, col
)

# Calculate the absolute and squared prediction errors.
# These help us measure how far predictions are from actual sales.
evaluated_df = (
    predictions_df
    .withColumn(
        "absolute_error",
        abs(col("units_sold") - col("predicted_units_sold"))
    )
    .withColumn(
        "squared_error",
        (col("units_sold") - col("predicted_units_sold")) ** 2
    )
)

# Summarize forecasting performance separately for each product.
product_metrics_df = evaluated_df.groupBy(
    "product_id", "product_name"
).agg(
    count("*").alias("test_records"),
    round(avg("units_sold"), 2).alias("average_actual_sales"),
    round(avg("predicted_units_sold"), 2).alias("average_predicted_sales"),
    round(avg("absolute_error"), 2).alias("mae"),
    round(sqrt(avg("squared_error")), 2).alias("rmse")
).orderBy("product_id")

# Display the product-level evaluation report.
display(product_metrics_df)


product_id,product_name,test_records,average_actual_sales,average_predicted_sales,mae,rmse
P001,Rice,61,29.84,29.91,2.34,2.9
P002,Cooking Oil,61,20.03,20.65,1.84,2.24
P003,Wheat Flour,61,24.87,25.1,2.35,2.71
P004,Sugar,61,18.11,19.01,1.81,2.22
P005,Milk,61,39.34,39.69,3.57,4.16


In [0]:

# ============================================================
# STEP 8B: ACTUAL VS PREDICTED DEMAND
# ============================================================

# Select one product to make the comparison easy to read.
# P001 represents Rice in our synthetic dataset.
rice_predictions_df = (
    predictions_df
    .filter(col("product_id") == "P001")
    .select(
        "sale_date",
        "units_sold",
        "predicted_units_sold"
    )
    .orderBy("sale_date")
)

# Display the results in Databricks.
display(rice_predictions_df)


sale_date,units_sold,predicted_units_sold
2026-08-01,34,34.190560568367935
2026-08-02,34,34.2128814491473
2026-08-03,28,32.111526068944166
2026-08-04,29,27.869999473322004
2026-08-05,33,31.441569233332206
2026-08-06,32,32.144968301048564
2026-08-07,34,30.54330165623664
2026-08-08,33,33.47511320883727
2026-08-09,31,33.36511114591687
2026-08-10,30,30.05680135573204



# Step 9: Inventory Optimization

## Objective
Convert demand forecasts into inventory recommendations.

## Business Concepts
- Lead time: Number of days required for a supplier to deliver stock.
- Safety stock: Extra inventory held to reduce stockout risk.
- Reorder point: Stock level at which a new order should be placed.
- Target stock: Recommended stock level to cover lead time and a review period.

## Approach
Use recent forecasting errors to estimate demand uncertainty,
then calculate safety stock, reorder points, and target inventory.

## Business Value
These recommendations can help reduce stockouts and avoid
holding unnecessarily high levels of inventory.


In [0]:

# ============================================================
# STEP 9: INVENTORY OPTIMIZATION
# ============================================================

from pyspark.sql.functions import (
    avg, stddev, ceil, greatest, lit, col, sqrt, round
)

# Business assumptions for this demonstration.
# Update these values to match your chosen retail scenario.
SUPPLIER_LEAD_TIME_DAYS = 5
REVIEW_PERIOD_DAYS = 7
SERVICE_LEVEL_Z = 1.65  # Approx. 95% one-sided service level

# Calculate demand statistics for each product using test predictions.
# These are demonstration estimates, not guaranteed future demand.
demand_stats_df = predictions_df.groupBy(
    "product_id", "product_name"
).agg(
    avg("predicted_units_sold").alias("avg_forecast_demand"),
    stddev("predicted_units_sold").alias("forecast_demand_stddev")
)

# Calculate the standard deviation of forecast errors per product.
# We use historical test errors as a simple estimate of uncertainty.
error_stats_df = (
    evaluated_df.groupBy("product_id")
    .agg(
        stddev("absolute_error").alias("error_stddev")
    )
)

# Join product demand statistics with forecast-error statistics.
inventory_df = (
    demand_stats_df
    .join(error_stats_df, on="product_id", how="left")
    .withColumn(
        "avg_forecast_demand",
        greatest(col("avg_forecast_demand"), lit(0.0))
    )
    .withColumn(
        "forecast_demand_stddev",
        greatest(
            coalesce(col("forecast_demand_stddev"), lit(0.0)),
            lit(0.0)
        )
    )
    .withColumn(
        "error_stddev",
        greatest(
            coalesce(col("error_stddev"), lit(0.0)),
            lit(0.0)
        )
    )
)

# Estimate safety stock using demand variability during lead time.
# This is a simplified policy for the learning project.
inventory_df = inventory_df.withColumn(
    "safety_stock",
    ceil(
        lit(SERVICE_LEVEL_Z)
        * sqrt(lit(SUPPLIER_LEAD_TIME_DAYS))
        * col("forecast_demand_stddev")
    )
)

# Calculate the reorder point.
# Reorder when available inventory reaches this estimated threshold.
inventory_df = inventory_df.withColumn(
    "reorder_point",
    ceil(
        col("avg_forecast_demand") * lit(SUPPLIER_LEAD_TIME_DAYS)
        + col("safety_stock")
    )
)

# Calculate target stock for the lead time plus review period.
inventory_df = inventory_df.withColumn(
    "target_stock",
    ceil(
        col("avg_forecast_demand")
        * lit(SUPPLIER_LEAD_TIME_DAYS + REVIEW_PERIOD_DAYS)
        + col("safety_stock")
    )
)

# Round average demand for easier reading.
inventory_df = inventory_df.withColumn(
    "avg_forecast_demand",
    round(col("avg_forecast_demand"), 2)
)

# Display the recommended inventory policy for each product.
display(
    inventory_df.select(
        "product_id",
        "product_name",
        "avg_forecast_demand",
        "safety_stock",
        "reorder_point",
        "target_stock"
    ).orderBy("product_id")
)


product_id,product_name,avg_forecast_demand,safety_stock,reorder_point,target_stock
P001,Rice,29.91,8,158,367
P002,Cooking Oil,20.65,5,109,253
P003,Wheat Flour,25.1,6,132,308
P004,Sugar,19.01,4,100,233
P005,Milk,39.69,12,211,489


In [0]:
from pyspark.sql.functions import (
    avg, stddev, ceil, greatest, lit, col,
    sqrt, round, coalesce
)


# Step 10: Inventory Analytics Dashboard

## Objective
Present demand forecasts and inventory recommendations
in a simple, business-friendly format.

## Key Metrics
- Average predicted daily demand
- Recommended safety stock
- Reorder point by product
- Target inventory level

## Visualizations
1. Predicted demand by product
2. Reorder point versus target stock
3. Inventory replenishment status

## Business Value
The dashboard helps inventory managers compare products
and make informed replenishment decisions.

Note: Inventory status depends on current stock quantities.
The demonstration will use sample inventory values.


In [0]:

# ============================================================
# STEP 10A: PREPARE DASHBOARD DATA
# ============================================================

from pyspark.sql.functions import when, col, lit

# Sample on-hand inventory for demonstration purposes.
# Replace these values with real stock data when available.
sample_stock = [
    ("P001", 180),
    ("P002", 90),
    ("P003", 150),
    ("P004", 100),
    ("P005", 220)
]

stock_df = spark.createDataFrame(
    sample_stock,
    ["product_id", "current_stock"]
)

# Combine inventory recommendations with current stock.
dashboard_df = inventory_df.join(
    stock_df,
    on="product_id",
    how="left"
)

# Determine whether each product needs replenishment.
dashboard_df = dashboard_df.withColumn(
    "inventory_status",
    when(
        col("current_stock") <= col("reorder_point"),
        lit("Reorder Required")
    ).otherwise(lit("Stock Available"))
)

# Estimate the quantity needed to reach target stock.
dashboard_df = dashboard_df.withColumn(
    "suggested_order_quantity",
    when(
        col("current_stock") < col("target_stock"),
        col("target_stock") - col("current_stock")
    ).otherwise(lit(0))
)

# Display the dashboard-ready dataset.
display(
    dashboard_df.select(
        "product_id",
        "product_name",
        "avg_forecast_demand",
        "current_stock",
        "safety_stock",
        "reorder_point",
        "target_stock",
        "inventory_status",
        "suggested_order_quantity"
    ).orderBy("product_id")
)


product_id,product_name,avg_forecast_demand,current_stock,safety_stock,reorder_point,target_stock,inventory_status,suggested_order_quantity
P001,Rice,29.91,180,8,158,367,Stock Available,187
P002,Cooking Oil,20.65,90,5,109,253,Reorder Required,163
P003,Wheat Flour,25.1,150,6,132,308,Stock Available,158
P004,Sugar,19.01,100,4,100,233,Reorder Required,133
P005,Milk,39.69,220,12,211,489,Stock Available,269



# Step 11: Store Results Using Delta Lake

## Objective
Save the prepared sales data, model predictions, and inventory
recommendations in Delta format.

## Tables Created
1. sales_history — Historical sales records
2. demand_predictions — Actual and predicted demand
3. inventory_recommendations — Inventory planning results

## Business Value
Delta tables allow the results to be reused for reporting,
analysis, and future machine learning workflows.

Note: These tables contain synthetic project data.


In [0]:

# ============================================================
# STEP 11: SAVE RESULTS AS DELTA TABLES
# ============================================================

# Use a dedicated schema for the project.
# This creates the schema if it does not already exist.
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.ai_demand_forecasting
""")

# Save historical sales data.
sales_history_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.ai_demand_forecasting.sales_history")

# Save model predictions.
predictions_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.ai_demand_forecasting.demand_predictions")

# Save inventory recommendations and dashboard fields.
dashboard_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.ai_demand_forecasting.inventory_recommendations")

print("All three Delta tables have been saved successfully!")


All three Delta tables have been saved successfully!


In [0]:

# Verify that the Delta tables are available.

spark.sql("""
SHOW TABLES IN workspace.ai_demand_forecasting
""").show(truncate=False)

# Check the number of historical sales records.
spark.sql("""
SELECT COUNT(*) AS total_sales_records
FROM workspace.ai_demand_forecasting.sales_history
""").show()

# Preview inventory recommendations.
display(
    spark.table(
        "workspace.ai_demand_forecasting.inventory_recommendations"
    )
)


+---------------------+-------------------------+-----------+
|database             |tableName                |isTemporary|
+---------------------+-------------------------+-----------+
|ai_demand_forecasting|demand_predictions       |false      |
|ai_demand_forecasting|inventory_recommendations|false      |
|ai_demand_forecasting|sales_history            |false      |
+---------------------+-------------------------+-----------+

+-------------------+
|total_sales_records|
+-------------------+
|               1825|
+-------------------+



product_id,product_name,avg_forecast_demand,forecast_demand_stddev,error_stddev,safety_stock,reorder_point,target_stock,current_stock,inventory_status,suggested_order_quantity
P001,Rice,29.91,2.057102119209289,1.7344667605699726,8,158,367,180,Stock Available,187
P002,Cooking Oil,20.65,1.3073171712895624,1.2839630551255163,5,109,253,90,Reorder Required,163
P003,Wheat Flour,25.1,1.440666694179064,1.3633693424829803,6,132,308,150,Stock Available,158
P004,Sugar,19.01,1.035296243857541,1.3005406494870841,4,100,233,100,Reorder Required,133
P005,Milk,39.69,3.0228339395439705,2.147678253091155,12,211,489,220,Stock Available,269
